# I22 USAXS processing through the MoDaCor runtime

This working notebook preprocesses each four-scan USAXS acquisition, reuses scans 977724--977727 as the empty-furnace background, and executes the correction graph in `pipelines/I22_USAXS.yaml`. The implementation rationale and open calibration choices are recorded in [`USAXS_PROCESSING_DESIGN.md`](USAXS_PROCESSING_DESIGN.md).


## Imports and example discovery


In [ ]:
from pathlib import Path
import atexit
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/I22")
sys.path.insert(0, str(PROJECT_DIR))

import h5py
import hdf5plugin
import matplotlib.pyplot as plt
import modacor
import numpy as np
from IPython.display import JSON, Markdown, display
from modacor.client import LocalRuntimeServer
from modacor.models.scaling import DependentData1D, prepare_scale_fit_data
from modacor.runner.pipeline import Pipeline

from usaxs_preprocess import (
    BACKGROUND_SCAN_SET,
    SAMPLE_SCAN_SETS,
    preprocess_usaxs_acquisition,
)


## Configuration

`MAX_MEASUREMENTS = 1` is useful for a quick smoke run. The high-gain front threshold, diode-response interval, and gain-overlap interval remain provisional, documented pipeline parameters.


In [ ]:
DATA_DIR = PROJECT_DIR / "data" / "usaxs"
PREPROCESSED_DIR = PROJECT_DIR / "work" / "preprocessed" / "usaxs"
OUTPUT_DIR = PROJECT_DIR / "work" / "output" / "usaxs"
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "I22_USAXS.yaml"
MAX_MEASUREMENTS = None
OVERWRITE_PREPROCESSED = False

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE = {"enabled": True, "watch": {"pooled": ["signal", "Q"]}}
OUTPUT_DATA_PATHS = [
    "/pooled/signal",
    "/pooled/Q",
    "/pooled/bin_id",
    "/pooled/bin_edges",
    "/pooled/bin_count",
    "/pooled/effective_sample_size",
    "/pooled/positive_weight_count",
    "/pooled/sum_weights",
    "/sample_integral/signal",
    "/background_integral/signal",
    "/transmission_factor/signal",
    "/SLF/beam_center",
    "/SHF/beam_center",
    "/S_science_low_rear/diode_scale_factor",
    "/S_science_low_rear/diode_fit_point_count",
    "/S_science_low_rear/diode_fit_x_min",
    "/S_science_low_rear/diode_fit_x_max",
    "/S_science_low_rear/diode_fit_reduced_chi_square",
    "/B_science_low_rear/diode_scale_factor",
    "/B_science_low_rear/diode_fit_point_count",
    "/B_science_low_rear/diode_fit_x_min",
    "/B_science_low_rear/diode_fit_x_max",
    "/B_science_low_rear/diode_fit_reduced_chi_square",
    "/SH/gain_scale_factor",
    "/SH/gain_fit_point_count",
    "/SH/gain_fit_x_min",
    "/SH/gain_fit_x_max",
    "/SH/gain_fit_reduced_chi_square",
    "/BH/gain_scale_factor",
    "/BH/gain_fit_point_count",
    "/BH/gain_fit_x_min",
    "/BH/gain_fit_x_max",
    "/BH/gain_fit_reduced_chi_square",
    "/sample_merged/signed_Q",
    "/sample_merged/both_wings_signal",
]
for key in ("SL", "SH", "BL", "BH"):
    OUTPUT_DATA_PATHS.extend(
        [f"/{key}/signal", f"/{key}/Q", f"/{key}/positive_weight_count"]
    )
for key in ("SLF", "SLR", "SHF", "SHR", "BLF", "BLR", "BHF", "BHR"):
    OUTPUT_DATA_PATHS.append(f"/{key}/signal_to_dark_noise")


## Lean preprocessing

The preprocessor reduces diode subreads and subtracts the matching exposure-scaled diode dark. I0 is summarized but receives no dark correction. For each diode readout, explicit copy and divide steps retain a dimensionless `signal_to_dark_noise` diagnostic before a separate threshold step creates the dynamic-range mask. MoDaCor then performs count-time and I0 normalization, response-scales and uncertainty-averages each aligned front/rear pair, and obtains one shared beam centre per gain scan. After Q conversion, the low-gain rear/front response is refitted near the direct beam and applied to both gains. Low and high merged-pair curves are then response-matched, jointly reduced, integrated for transmission, and background-subtracted.


In [ ]:
background_path = preprocess_usaxs_acquisition(
    DATA_DIR, BACKGROUND_SCAN_SET, PREPROCESSED_DIR, overwrite=OVERWRITE_PREPROCESSED
)
sample_paths = [
    preprocess_usaxs_acquisition(
        DATA_DIR, scan_set, PREPROCESSED_DIR, overwrite=OVERWRITE_PREPROCESSED
    )
    for scan_set in SAMPLE_SCAN_SETS[:MAX_MEASUREMENTS]
]

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Background: {background_path.name}")
print(f"Samples: {len(sample_paths)}")


## Inspect the correction graph


In [ ]:
pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
pipeline.prepare()
display(Markdown(f"```mermaid\n{pipeline.to_mermaid(direction='TD')}\n```"))
print(f"USAXS pipeline: {len(pipeline.graph)} steps")


## Start or reuse a local runtime


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=OUTPUT_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Process the three sample acquisitions

The background source stays registered while only the sample source changes, allowing the runtime to use its dependency graph for partial reruns.


In [ ]:
session = client.replace_session(
    "i22-usaxs-server-batch",
    name="I22 USAXS server batch",
    pipeline_yaml_path=str(PIPELINE_PATH),
    trace=TRACE,
)
session.register_sources(
    {"ref": "sample", "type": "hdf", "location": str(sample_paths[0])},
    {"ref": "background", "type": "hdf", "location": str(background_path)},
)

results = []
for sample_path in sample_paths:
    output = OUTPUT_DIR / f"{sample_path.stem}_result.h5"
    session.register_source({"ref": "sample", "type": "hdf", "location": str(sample_path)})
    result = session.process(
        mode="auto",
        changed_sources=["sample"],
        run_name=sample_path.stem,
        rollback_snapshot=False,
        write_hdf={"path": str(output), "data_paths": OUTPUT_DATA_PATHS},
    )
    results.append({"sample": sample_path, "output": output, "result": result})
    print(f"{sample_path.name}: {result['status']} ({result['effective_mode']})")

display(JSON(session.latest_error()))


## Compare reduced curves and diagnostics

The stored Q values are the actual weighted mean coordinates of contributing points, not geometric bin centres. Negative intensities remain visible because they are meaningful background-subtraction diagnostics. The second plot checks the scaled low/high overlap using the explicitly selected `subread_sem` component; it does not treat the final sample scattering as a residual expected to equal zero.


In [ ]:
def hdf_dependent_1d(run, bundle_key, uncertainty_key="subread_sem"):
    signal = run[f"{bundle_key}/signal"]
    axis = run[f"{bundle_key}/Q"]
    y = np.asarray(signal["signal"][()], dtype=float)
    q = np.asarray(axis["signal"][()], dtype=float)
    sigma = np.broadcast_to(
        np.asarray(signal[f"uncertainties/{uncertainty_key}"][()], dtype=float), y.shape
    )
    weights = np.asarray(signal["weights"][()], dtype=float) if "weights" in signal else np.ones_like(y)
    return q, DependentData1D(y=y, sigma=sigma, weights=np.broadcast_to(weights, y.shape))


def gain_overlap_residuals(run, low_key, high_key):
    q_low, low = hdf_dependent_1d(run, low_key)
    q_high, high = hdf_dependent_1d(run, high_key)
    aligned = prepare_scale_fit_data(
        x_work=q_high,
        dep_work=high,
        x_ref=q_low,
        dep_ref=low,
        require_overlap=True,
        interpolation_kind="linear",
        fit_min=-0.005,
        fit_max=-0.002,
        use_weights=True,
    )
    sigma = np.sqrt(aligned.sigma_ref**2 + aligned.sigma_work**2)
    return aligned.x, (aligned.y_ref - aligned.y_work) / sigma


def longest_true_run(values):
    longest = current = 0
    for value in values:
        current = current + 1 if value else 0
        longest = max(longest, current)
    return longest


fig, (ax_wings, ax_curve, ax_count) = plt.subplots(3, 1, figsize=(8, 11))
fig_overlap, ax_overlap = plt.subplots(figsize=(8, 4))
background_overlap_plotted = False
for item in results:
    with h5py.File(item["output"], "r") as source:
        result_root = source["/processing/result"]
        run = result_root[next(iter(result_root))]
        q = run["pooled/Q/signal"][()]
        intensity = run["pooled/signal/signal"][()]
        count = run["pooled/bin_count/signal"][()]
        signed_q = run["sample_merged/signed_Q/signal"][()]
        both_wings = run["sample_merged/both_wings_signal/signal"][()]
        transmission = float(np.asarray(run["transmission_factor/signal/signal"][()]).squeeze())
        sample_integral = float(np.asarray(run["sample_integral/signal/signal"][()]).squeeze())
        background_integral = float(np.asarray(run["background_integral/signal/signal"][()]).squeeze())
        low_center = float(np.asarray(run["SLF/beam_center/signal"][()]).squeeze())
        high_center = float(np.asarray(run["SHF/beam_center/signal"][()]).squeeze())
        diode_scales = [
            float(np.asarray(run[f"{key}_science_low_rear/diode_scale_factor/signal"][()]).squeeze())
            for key in ("S", "B")
        ]
        diode_chi2 = [
            float(np.asarray(run[f"{key}_science_low_rear/diode_fit_reduced_chi_square/signal"][()]).squeeze())
            for key in ("S", "B")
        ]
        gain_scales = [
            float(np.asarray(run[f"{key}/gain_scale_factor/signal"][()]).squeeze())
            for key in ("SH", "BH")
        ]
        gain_chi2 = [
            float(np.asarray(run[f"{key}/gain_fit_reduced_chi_square/signal"][()]).squeeze())
            for key in ("SH", "BH")
        ]
        sample_overlap_q, sample_overlap_z = gain_overlap_residuals(run, "SL", "SH")
        background_overlap_q, background_overlap_z = gain_overlap_residuals(run, "BL", "BH")
    label = item["sample"].stem
    order = np.argsort(signed_q, kind="stable")
    ax_wings.plot(signed_q[order], both_wings[order], label=label)
    ax_curve.plot(q, intensity, label=label)
    ax_count.plot(q, count, label=label)
    ax_overlap.plot(-sample_overlap_q, sample_overlap_z, alpha=0.7, label=f"{label} sample")
    if not background_overlap_plotted:
        ax_overlap.plot(
            -background_overlap_q, background_overlap_z, color="black", alpha=0.55, label="background"
        )
        background_overlap_plotted = True
    q_order = np.argsort(q, kind="stable")
    negative = intensity[q_order] < 0
    print(
        f"{label}: T={transmission:.6f} ({sample_integral:.5g}/{background_integral:.5g}), "
        f"centres={low_center:.4f}/{high_center:.4f} urad, "
        f"rear/front S/B={diode_scales[0]:.5g}/{diode_scales[1]:.5g} "
        f"(chi2={diode_chi2[0]:.3g}/{diode_chi2[1]:.3g}), "
        f"high/low S/B={gain_scales[0]:.5g}/{gain_scales[1]:.5g} "
        f"(chi2={gain_chi2[0]:.3g}/{gain_chi2[1]:.3g}), "
        f"negative bins={negative.sum()}/{negative.size}, longest run={longest_true_run(negative)}"
    )

ax_wings.axvline(0, color="0.6", linewidth=0.8)
ax_wings.axhline(0, color="0.6", linewidth=0.8)
ax_wings.set_xscale("symlog", linthresh=8e-4)
ax_wings.set_yscale("symlog", linthresh=1e-5)
ax_wings.set_xlabel(r"signed $Q$ / nm$^{-1}$")
ax_wings.set_ylabel("merged, corrected both-wing intensity")
ax_wings.legend()
ax_curve.axhline(0, color="0.6", linewidth=0.8)
ax_curve.set_xscale("log")
ax_curve.set_xlabel(r"$Q$ / nm$^{-1}$")
ax_curve.set_ylabel("corrected intensity")
ax_curve.legend()
ax_count.set_xscale("log")
ax_count.set_yscale("log")
ax_count.set_xlabel(r"$Q$ / nm$^{-1}$")
ax_count.set_ylabel("contributing points")
fig.tight_layout()
ax_overlap.axhline(0, color="0.5", linewidth=0.8)
ax_overlap.set_xscale("log")
ax_overlap.set_xlabel(r"negative-wing $Q$ / nm$^{-1}$")
ax_overlap.set_ylabel(r"$(I_{low} - I_{high}) / \sigma_{subread}$")
ax_overlap.set_title("Scaled low/high overlap; no hard handoff")
ax_overlap.legend()
fig_overlap.tight_layout()


## Cleanup

This stops only a process launched by this notebook. A runtime already listening on the configured port is left alone.


In [ ]:
server.stop()
print("Stopped the notebook-owned runtime." if not client.is_ready() else "Left the external runtime running.")
